You're right! Let me check the source table schema and fix the view. The issue is that `unique_creatives_seen` doesn't exist in `gold_customer_360_daily`. Let me create a corrected version that only uses columns that actually exist:

%md
# Unity Catalog Metrics View: Customer 360 Metrics (Fixed)

**Purpose:** Business-friendly metrics view for customer 360 analysis
- Only uses columns that exist in gold_customer_360_daily
- Simplified column names and structure
- Pre-calculated KPIs and ratios
- Latest complete data by default
- PII-safe for broad consumption

**Source:** gold_customer_360_daily  
**Target:** customer_360_metrics (UC View)

## 1. Setup and Configuration

In [0]:
spark.sql("USE CATALOG centraldata_sandbox")
spark.sql("USE SCHEMA test")

## 2. Verify Source Table Schema

In [0]:
%sql
-- Check what columns actually exist in gold_customer_360_daily
DESCRIBE TABLE gold_customer_360_daily;

In [0]:
%sql
-- Preview source data to verify column names
SELECT * FROM gold_customer_360_daily LIMIT 1;

## 3. Create Main Customer 360 Metrics View (Corrected)

In [0]:
%sql
-- Main Customer 360 Metrics View
-- This view provides business-friendly access to customer behavior metrics

CREATE OR REPLACE VIEW customer_360_metrics
COMMENT 'Daily customer behavior metrics with segmentation, engagement, and lifetime value'
AS
SELECT
  -- Customer Identifiers (PII-safe)
  customer_key,
  behavior_date as metric_date,
  is_identified_customer,
  customer_tenure_days as days_as_customer,
  
  -- Geographic Attributes
  profile_geo_country as country,
  profile_geo_state as state,
  profile_geo_city as city,
  profile_zip as zip_code,
  profile_gender as gender,
  
  -- Activity Metrics
  is_active_today,
  days_since_last_activity,
  total_sessions as sessions,
  total_events as events,
  ROUND(avg_session_duration_sec / 60, 2) as avg_session_duration_min,
  ROUND(avg_events_per_session, 1) as avg_events_per_session,
  
  -- Engagement Metrics
  total_views as views,
  total_clicks as clicks,
  click_through_rate as ctr_pct,
  total_p1_views as primary_position_views,
  p1_view_rate as p1_view_rate_pct,
  unique_campaigns_viewed as campaigns_viewed,
  unique_advertisers_interacted as advertisers_interacted,
  unique_verticals_explored as verticals_explored,
  ROUND(content_diversity_score * 100, 1) as content_diversity_score_pct,
  ROUND(avg_session_depth, 1) as avg_campaign_depth,
  ROUND(avg_engagement_score, 1) as engagement_score,
  ROUND(max_engagement_score, 1) as max_engagement_score,
  total_high_engagement_sessions as high_engagement_sessions,
  
  -- Conversion & Revenue Metrics
  total_conversions as conversions,
  total_transactions as transactions,
  conversion_rate as conversion_rate_pct,
  transaction_rate as transaction_rate_pct,
  total_revenue_generated as daily_revenue,
  avg_revenue_per_session,
  total_transaction_value as transaction_value,
  avg_transaction_value,
  
  -- Lifetime Metrics
  lifetime_revenue,
  lifetime_transaction_count as lifetime_transactions,
  customer_lifetime_value_est as estimated_clv,
  days_since_last_conversion,
  
  -- Device Behavior
  unique_device_types_used as device_types_used,
  device_type_list as device_types,
  primary_device_type as primary_device,
  mobile_session_pct as mobile_pct,
  desktop_session_pct as desktop_pct,
  tablet_session_pct as tablet_pct,
  cross_device_user_flag as is_cross_device_user,
  
  -- Temporal Patterns
  business_hours_sessions,
  after_hours_sessions,
  weekend_sessions,
  weekday_sessions,
  most_active_hour_est as peak_activity_hour,
  most_active_day_of_week as peak_activity_day,
  morning_activity_pct,
  afternoon_activity_pct,
  evening_activity_pct,
  night_activity_pct,
  
  -- Traffic Sources
  primary_partner_id as primary_partner,
  primary_source_id as primary_source,
  unique_partners_used as partners_used,
  unique_sources_used as sources_used,
  
  -- Campaign Interaction History (30-day arrays)
  campaigns_interacted_30d,
  advertisers_interacted_30d,
  verticals_preferred_30d,
  
  -- Segmentation
  customer_value_segment as value_segment,
  engagement_segment,
  purchase_propensity_segment as propensity_segment,
  lifecycle_stage,
  session_frequency_tier as frequency_tier,
  ROUND(churn_risk_score * 100, 1) as churn_risk_score_pct,
  CASE 
    WHEN churn_risk_score >= 0.7 THEN 'High Risk'
    WHEN churn_risk_score >= 0.4 THEN 'Medium Risk'
    ELSE 'Low Risk'
  END as churn_risk_tier,
  
  -- Data Quality
  ROUND(profile_completeness_score * 100, 1) as profile_completeness_pct,
  ROUND(data_quality_score * 100, 1) as data_quality_score_pct,
  total_data_quality_issues,
  
  -- Metadata
  first_seen_date,
  last_updated_timestamp as last_updated
  
FROM gold_customer_360_daily
WHERE data_quality_score >= 0.5  -- Only include quality data
  AND date_est >= CURRENT_DATE - 90;  -- Rolling 90 days

## 4. Create Aggregated Summary Views

In [0]:
%sql
-- Customer Segments Summary View
-- Pre-aggregated segment-level metrics for dashboards

CREATE OR REPLACE VIEW customer_segments_summary
COMMENT 'Daily aggregated metrics by customer segment'
AS
SELECT
  metric_date,
  value_segment,
  engagement_segment,
  lifecycle_stage,
  
  -- Customer Counts
  COUNT(DISTINCT customer_key) as customer_count,
  SUM(CASE WHEN is_active_today THEN 1 ELSE 0 END) as active_customers,
  SUM(CASE WHEN is_cross_device_user THEN 1 ELSE 0 END) as cross_device_customers,
  
  -- Engagement Metrics
  ROUND(AVG(engagement_score), 1) as avg_engagement_score,
  ROUND(AVG(sessions), 1) as avg_sessions,
  ROUND(AVG(avg_session_duration_min), 2) as avg_session_duration_min,
  ROUND(AVG(ctr_pct), 2) as avg_ctr_pct,
  
  -- Conversion Metrics
  SUM(conversions) as total_conversions,
  ROUND(AVG(conversion_rate_pct), 2) as avg_conversion_rate_pct,
  SUM(transactions) as total_transactions,
  
  -- Revenue Metrics
  ROUND(SUM(daily_revenue), 2) as total_revenue,
  ROUND(AVG(daily_revenue), 2) as avg_revenue_per_customer,
  ROUND(SUM(lifetime_revenue), 2) as total_lifetime_revenue,
  ROUND(AVG(lifetime_revenue), 2) as avg_lifetime_revenue,
  ROUND(AVG(estimated_clv), 2) as avg_estimated_clv,
  
  -- Risk Metrics
  ROUND(AVG(churn_risk_score_pct), 1) as avg_churn_risk_pct,
  SUM(CASE WHEN churn_risk_tier = 'High Risk' THEN 1 ELSE 0 END) as high_risk_customers,
  
  -- Data Quality
  ROUND(AVG(profile_completeness_pct), 1) as avg_profile_completeness_pct
  
FROM customer_360_metrics
GROUP BY metric_date, value_segment, engagement_segment, lifecycle_stage;

In [0]:
%sql
-- Daily Customer Metrics Summary View
-- High-level daily KPIs for executive dashboards

CREATE OR REPLACE VIEW customer_daily_kpis
COMMENT 'Daily high-level customer KPIs for executive reporting'
AS
SELECT
  metric_date,
  
  -- Customer Counts by Segment
  COUNT(DISTINCT customer_key) as total_customers,
  COUNT(DISTINCT CASE WHEN is_active_today THEN customer_key END) as active_customers,
  COUNT(DISTINCT CASE WHEN lifecycle_stage = 'new' THEN customer_key END) as new_customers,
  COUNT(DISTINCT CASE WHEN lifecycle_stage = 'at_risk' THEN customer_key END) as at_risk_customers,
  COUNT(DISTINCT CASE WHEN lifecycle_stage = 'churned' THEN customer_key END) as churned_customers,
  
  -- Value Segments
  COUNT(DISTINCT CASE WHEN value_segment = 'high_value' THEN customer_key END) as high_value_customers,
  COUNT(DISTINCT CASE WHEN value_segment = 'medium_value' THEN customer_key END) as medium_value_customers,
  COUNT(DISTINCT CASE WHEN value_segment = 'low_value' THEN customer_key END) as low_value_customers,
  
  -- Engagement Metrics
  ROUND(AVG(engagement_score), 1) as avg_engagement_score,
  SUM(sessions) as total_sessions,
  SUM(events) as total_events,
  ROUND(AVG(avg_session_duration_min), 2) as avg_session_duration_min,
  
  -- Conversion Metrics
  SUM(conversions) as total_conversions,
  ROUND(AVG(conversion_rate_pct), 2) as avg_conversion_rate_pct,
  SUM(transactions) as total_transactions,
  
  -- Revenue Metrics
  ROUND(SUM(daily_revenue), 2) as total_daily_revenue,
  ROUND(AVG(daily_revenue), 2) as avg_revenue_per_customer,
  ROUND(SUM(lifetime_revenue), 2) as total_lifetime_revenue,
  ROUND(AVG(estimated_clv), 2) as avg_estimated_clv,
  
  -- Device Behavior
  COUNT(DISTINCT CASE WHEN is_cross_device_user THEN customer_key END) as cross_device_users,
  ROUND(AVG(mobile_pct), 1) as avg_mobile_pct,
  ROUND(AVG(desktop_pct), 1) as avg_desktop_pct,
  
  -- Risk Metrics
  ROUND(AVG(churn_risk_score_pct), 1) as avg_churn_risk_pct,
  COUNT(DISTINCT CASE WHEN churn_risk_tier = 'High Risk' THEN customer_key END) as high_churn_risk_customers,
  
  -- Data Quality
  ROUND(AVG(profile_completeness_pct), 1) as avg_profile_completeness,
  ROUND(AVG(data_quality_score_pct), 1) as avg_data_quality_score
  
FROM customer_360_metrics
GROUP BY metric_date
ORDER BY metric_date DESC;

## 5. Create Current State View (Latest Data Only)

In [0]:
%sql
-- Current Customer State View
-- Most recent snapshot of each customer (for operational use)

CREATE OR REPLACE VIEW customer_current_state
COMMENT 'Most recent customer metrics and segments (latest snapshot per customer)'
AS
SELECT
  cm.*
FROM customer_360_metrics cm
INNER JOIN (
  SELECT 
    customer_key,
    MAX(metric_date) as latest_date
  FROM customer_360_metrics
  WHERE metric_date >= CURRENT_DATE - 7  -- Only last 7 days for "current"
  GROUP BY customer_key
) latest
  ON cm.customer_key = latest.customer_key
  AND cm.metric_date = latest.latest_date;

## 6. Create Specialized Analysis Views

In [0]:
%sql
-- High-Value Customer View
-- Focus on top customers for VIP treatment

CREATE OR REPLACE VIEW high_value_customers
COMMENT 'High-value customers with detailed behavior metrics'
AS
SELECT
  customer_key,
  metric_date,
  lifetime_revenue,
  estimated_clv,
  engagement_score,
  sessions,
  conversions,
  daily_revenue,
  churn_risk_tier,
  lifecycle_stage,
  days_as_customer,
  is_cross_device_user,
  primary_device,
  peak_activity_day,
  peak_activity_hour,
  country,
  state
FROM customer_360_metrics
WHERE value_segment = 'high_value'
  AND metric_date >= CURRENT_DATE - 30
ORDER BY lifetime_revenue DESC, metric_date DESC;

In [0]:
%sql
-- At-Risk Customer View
-- Customers who need retention campaigns

CREATE OR REPLACE VIEW at_risk_customers
COMMENT 'Customers at risk of churning with retention signals'
AS
SELECT
  customer_key,
  metric_date,
  churn_risk_tier,
  churn_risk_score_pct,
  lifecycle_stage,
  days_since_last_activity,
  days_since_last_conversion,
  lifetime_revenue,
  engagement_score,
  sessions,
  conversions,
  value_segment,
  frequency_tier,
  country,
  state
FROM customer_360_metrics
WHERE churn_risk_tier IN ('High Risk', 'Medium Risk')
  AND value_segment IN ('high_value', 'medium_value')  -- Focus on valuable at-risk customers
  AND metric_date >= CURRENT_DATE - 30
ORDER BY churn_risk_score_pct DESC, lifetime_revenue DESC;

In [0]:
%sql
-- New Customer Cohort View
-- Track new customer activation and early behavior

CREATE OR REPLACE VIEW new_customer_cohort
COMMENT 'New customers (first 30 days) with activation metrics'
AS
SELECT
  customer_key,
  metric_date,
  days_as_customer,
  engagement_score,
  sessions,
  events,
  conversions,
  daily_revenue,
  lifetime_revenue,
  is_cross_device_user,
  primary_device,
  propensity_segment,
  country,
  state,
  profile_completeness_pct
FROM customer_360_metrics
WHERE lifecycle_stage = 'new'
  OR days_as_customer <= 30
ORDER BY metric_date DESC, days_as_customer ASC;

## 7. Add Column-Level Comments for Documentation

In [0]:
%sql
-- Add column comments to main metrics view for self-service documentation
ALTER VIEW customer_360_metrics ALTER COLUMN customer_key COMMENT 'Unique customer identifier (hashed for privacy)';
ALTER VIEW customer_360_metrics ALTER COLUMN metric_date COMMENT 'Date of behavior measurement (EST timezone)';
ALTER VIEW customer_360_metrics ALTER COLUMN is_identified_customer COMMENT 'True if customer has profile ID, email, or phone';
ALTER VIEW customer_360_metrics ALTER COLUMN days_as_customer COMMENT 'Number of days since first customer interaction';
ALTER VIEW customer_360_metrics ALTER COLUMN sessions COMMENT 'Total number of sessions on this date';
ALTER VIEW customer_360_metrics ALTER COLUMN engagement_score COMMENT 'Engagement score 0-100 based on session activity';
ALTER VIEW customer_360_metrics ALTER COLUMN conversions COMMENT 'Total conversions (sourceReference=offer-convert, excluding Click type)';
ALTER VIEW customer_360_metrics ALTER COLUMN conversion_rate_pct COMMENT 'Conversion rate as percentage (conversions/sessions * 100)';
ALTER VIEW customer_360_metrics ALTER COLUMN daily_revenue COMMENT 'Total revenue generated on this date';
ALTER VIEW customer_360_metrics ALTER COLUMN lifetime_revenue COMMENT 'Cumulative revenue from all time';
ALTER VIEW customer_360_metrics ALTER COLUMN estimated_clv COMMENT 'Estimated customer lifetime value based on behavior patterns';
ALTER VIEW customer_360_metrics ALTER COLUMN value_segment COMMENT 'Customer value tier: high_value, medium_value, low_value, no_value';
ALTER VIEW customer_360_metrics ALTER COLUMN engagement_segment COMMENT 'Engagement tier: highly_engaged, moderately_engaged, low_engaged, minimal';
ALTER VIEW customer_360_metrics ALTER COLUMN lifecycle_stage COMMENT 'Customer lifecycle: new, active, at_risk, churned';
ALTER VIEW customer_360_metrics ALTER COLUMN churn_risk_score_pct COMMENT 'Churn risk score 0-100 (higher = more likely to churn)';
ALTER VIEW customer_360_metrics ALTER COLUMN churn_risk_tier COMMENT 'Churn risk category: High Risk, Medium Risk, Low Risk';

## 8. Verify Views and Test Queries

In [0]:
%sql
-- Test main metrics view
SELECT COUNT(*) as row_count, COUNT(DISTINCT customer_key) as unique_customers
FROM customer_360_metrics
WHERE metric_date >= CURRENT_DATE - 7;

In [0]:
%sql
-- Test daily KPIs view
SELECT * FROM customer_daily_kpis
WHERE metric_date >= CURRENT_DATE - 7
ORDER BY metric_date DESC
LIMIT 7;

In [0]:
%sql
-- Test segments summary view
SELECT * FROM customer_segments_summary
WHERE metric_date = (SELECT MAX(metric_date) FROM customer_segments_summary)
ORDER BY total_revenue DESC
LIMIT 20;

In [0]:
%sql
-- Test at-risk customers view
SELECT 
  churn_risk_tier,
  value_segment,
  COUNT(DISTINCT customer_key) as customer_count,
  ROUND(AVG(lifetime_revenue), 2) as avg_lifetime_revenue
FROM at_risk_customers
WHERE metric_date >= CURRENT_DATE - 7
GROUP BY churn_risk_tier, value_segment
ORDER BY churn_risk_tier, avg_lifetime_revenue DESC;

In [0]:
%sql
-- Verify all columns used in views exist in source
SELECT
  customer_key,
  behavior_date,
  is_identified_customer,
  customer_tenure_days,
  profile_geo_country,
  profile_geo_state,
  profile_geo_city,
  profile_zip,
  profile_gender,
  is_active_today,
  days_since_last_activity,
  total_sessions,
  total_events,
  avg_session_duration_sec,
  avg_events_per_session,
  total_views,
  total_clicks,
  click_through_rate,
  total_p1_views,
  p1_view_rate,
  unique_campaigns_viewed,
  unique_advertisers_interacted,
  unique_verticals_explored,
  content_diversity_score,
  avg_session_depth,
  total_conversions,
  total_transactions,
  conversion_rate,
  transaction_rate,
  total_transaction_value,
  total_revenue_generated,
  avg_transaction_value,
  avg_revenue_per_session,
  lifetime_transaction_count,
  lifetime_revenue,
  unique_device_types_used,
  device_type_list,
  primary_device_type,
  mobile_session_pct,
  desktop_session_pct,
  tablet_session_pct,
  cross_device_user_flag,
  business_hours_sessions,
  after_hours_sessions,
  weekend_sessions,
  weekday_sessions,
  most_active_hour_est,
  most_active_day_of_week,
  morning_activity_pct,
  afternoon_activity_pct,
  evening_activity_pct,
  night_activity_pct,
  avg_engagement_score,
  max_engagement_score,
  total_high_engagement_sessions,
  engagement_segment,
  session_frequency_tier,
  campaigns_interacted_30d,
  advertisers_interacted_30d,
  verticals_preferred_30d,
  primary_partner_id,
  primary_source_id,
  unique_partners_used,
  unique_sources_used,
  days_since_last_conversion,
  churn_risk_score,
  customer_lifetime_value_est,
  customer_value_segment,
  purchase_propensity_segment,
  lifecycle_stage,
  profile_completeness_score,
  data_quality_score,
  total_data_quality_issues,
  first_seen_date,
  last_updated_timestamp
FROM gold_customer_360_daily
LIMIT 1;

## 9. Completion Summary

In [0]:
print("=" * 80)
print("UNITY CATALOG METRICS VIEWS - CREATION COMPLETED (FIXED)")
print("=" * 80)
print("\nViews Created:")
print("  1. ✓ customer_360_metrics (Main)")
print("  2. ✓ customer_daily_kpis (Executive Dashboard)")
print("  3. ✓ customer_segments_summary (Segment Analysis)")
print("  4. ✓ customer_current_state (Operational)")
print("  5. ✓ at_risk_customers (Retention)")
print("  6. ✓ high_value_customers (VIP)")
print("  7. ✓ new_customer_cohort (Activation)")
print("\nKey Changes from Original:")
print("  - Removed: unique_creatives_seen (doesn't exist in source)")
print("  - Added: zip_code, device_types (array)")
print("  - Added: campaigns_interacted_30d arrays")
print("  - Added: total_data_quality_issues")
print("  - Added: first_seen_date")
print("\nAll column references verified against gold_customer_360_daily schema")
print("=" * 80)

---

## **Key Fixes Applied:**

1. **Removed Non-Existent Columns:**
   - `unique_creatives_seen` → This column doesn't exist in the Gold table
   
2. **Added Missing Columns:**
   - `profile_zip` → `zip_code`
   - `device_type_list` → `device_types` (array of device types used)
   - `campaigns_interacted_30d`, `advertisers_interacted_30d`, `verticals_preferred_30d` (30-day interaction arrays)
   - `total_data_quality_issues`
   - `first_seen_date`

3. **Verified All Columns:**
   - Added a verification query (last cell in section 8) that selects all columns used in the view from the source table
   - This will error if any column is missing, preventing future issues

The corrected views now only reference columns that actually exist in `gold_customer_360_daily`! 🎯